# Figure 2: Genetic regulation of the human brain glycoproteome (MAF >= 0.05)

In [ ]:
library(tidyverse)
library(data.table)
library(ggplot2)
library(patchwork)
library(RColorBrewer)
library(colorspace)
library(scales)
library(UpSetR)
library(ggrepel)
library(tidytext)

## prepare data

In [ ]:
exp_res_unadj <- fread('data_output/finemapping/ROSMAP_gpQTL.unadjusted.exported.toploci.bed') %>% mutate(gpID = str_extract(event_ID, 'gp_[0-9]+')) %>% filter(MAF >= 0.05)
exp_res_adj <- fread('data_output/finemapping/ROSMAP_gpQTL.adjusted.exported.toploci.bed') %>% mutate(gpID = str_extract(event_ID, 'gp_[0-9]+')) %>% filter(MAF >= 0.05)
exp_res <- rbind(exp_res_unadj, exp_res_adj)

gp_mapping <- fread('gp_coordinates_clean.txt')

# re map gpID to gene ID
exp_res <- exp_res %>% merge(., gp_mapping %>% select(ID, ensembl_id), by.x = c('gpID'), by.y = 'ID') %>% select(-gene_ID) %>% rename(gene_ID = ensembl_id)

qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')

annotation <- readRDS('rosmap/glycoproteomics/dlpfcTissue/batch1/annotation/gpAnnotation.Rds') %>%
  filter(!gpID %in% qc_remove_gpids) %>%
  # # the corrected file dropped Remove entirely
  mutate(Remove = FALSE) %>%
  mutate(GlycanTypeSummary = case_when(
    str_detect(GlycanType, "(^|;)gc(;|$)") & str_detect(GlycanType, "(^|;)N(;|$)") ~ "mixed",
    str_detect(GlycanType, "(^|;)gc(;|$)") ~ "glycation",
    str_detect(GlycanType, "(^|;)N(;|$)")  ~ "glycosylation",
    TRUE ~ GlycanTypeSummary
  ))

annotation <- annotation %>% filter(!gpID %in% w_sole_exclude_gpids)
annotation %>% head
annotation %>% dim

# exclude gpIDs whose fine-mapped signal is explained by a coding variant inside
coding_drop_gpids <- readLines('../../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt')
stopifnot(length(coding_drop_gpids) == 21)
annotation <- annotation %>% filter(!gpID %in% coding_drop_gpids)
annotation %>% dim

exp_res_anno <- merge(exp_res, annotation)

## Figure 2a

In [ ]:
glycan_colors <- c(
  "glycosylation" = "#4C72B0",
  "glycation"     = "#C44E52"
)

df_qtl <- exp_res_anno %>%
  filter(!Remove) %>%
  filter(cs_coverage_0.95 > 0, MAF >= 0.05) %>%
  mutate(
    cs_id = paste(event_ID, cs_coverage_0.95, sep = '_'),
    Adjustment = ifelse(grepl("unadjusted", event_ID), "Unadjusted", "Adjusted")
  ) %>%
  distinct(gpID, cs_id, GlycanTypeSummary, Adjustment)

qtl_counts <- df_qtl %>%
  group_by(Adjustment, GlycanTypeSummary) %>%
  summarise(NumGpID = n_distinct(gpID), .groups = "drop")
qtl_counts$Adjustment <- factor(qtl_counts$Adjustment, levels = c("Unadjusted", "Adjusted"))
dir.create('data')
saveRDS(qtl_counts, 'data/Figure_2a.rds')

# Table for 2a: number of 95% CSs
qtl_cs_table <- df_qtl %>%
  group_by(Adjustment, GlycanTypeSummary) %>%
  summarise(n_gpID = n_distinct(gpID), n_CS = n_distinct(gpID, cs_id), .groups = "drop") %>%
  mutate(CS_per_gpID = round(n_CS / n_gpID, 2)) %>%
  arrange(factor(Adjustment, levels = c("Unadjusted", "Adjusted")), GlycanTypeSummary)
stopifnot(identical(qtl_cs_table$n_gpID[order(qtl_cs_table$Adjustment, qtl_cs_table$GlycanTypeSummary)], qtl_counts$NumGpID[order(as.character(qtl_counts$Adjustment), qtl_counts$GlycanTypeSummary)]))
saveRDS(qtl_cs_table, 'data/Figure_2a_cs_table.rds')
fwrite(qtl_cs_table, 'data/Figure_2a_cs_table.tsv', sep = '\t')
as.data.frame(qtl_cs_table)

## Figure 2b

In [ ]:
# v202609 = v202505 with the TMEM106B rows replaced by the revised gp_8489-8491
unadj <- fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/phenotype_preprocessing/unadj_glyco.sorted.bed.gz')
used_gpid <- unadj %>% pull(ID) %>% unique
used_gpid %>% length

setdiff(annotation$gpID, used_gpid) %>% uniqueN

### 2b: gpID-level QTL enrichment by main glycan type

In [ ]:

# --- Universe: gpID-level QTL indicator and CS counts

qtl_gpid_unadj <- exp_res_anno %>%
  filter(!Remove, cs_coverage_0.95 > 0, grepl("unadjusted", event_ID)) %>%
  pull(gpID) %>%
  unique()

gp_cs_count <- exp_res_anno %>%
  filter(!Remove, cs_coverage_0.95 > 0, grepl("unadjusted", event_ID)) %>%
  mutate(cs_id = paste(event_ID, cs_coverage_0.95, sep = "_")) %>%
  distinct(gpID, cs_id) %>%
  count(gpID, name = "n_cs")

gp_set_main <- annotation %>%
  filter(GlycanTypeSummary != 'mixed') %>%
  filter(gpID %in% used_gpid) %>%
  distinct(gpID, GlycanTypeSummary) %>%
  left_join(tibble(gpID = qtl_gpid_unadj, has_qtl = TRUE), by = "gpID") %>%
  left_join(gp_cs_count, by = "gpID") %>%
  mutate(
    has_qtl = dplyr::coalesce(has_qtl, FALSE),
    n_cs    = tidyr::replace_na(n_cs, 0L)
  )

main_class_enrich_gp <- purrr::map_dfr(c("glycosylation", "glycation"), function(gt) {
  in_class <- gp_set_main$GlycanTypeSummary == gt
  mat <- matrix(
    c(
      sum( in_class &  gp_set_main$has_qtl),
      sum(!in_class &  gp_set_main$has_qtl),
      sum( in_class & !gp_set_main$has_qtl),
      sum(!in_class & !gp_set_main$has_qtl)
    ),
    nrow = 2, byrow = TRUE,
    dimnames = list(QTL = c("WithQTL", "WithoutQTL"), Type = c(gt, "Others"))
  )
  ft <- fisher.test(mat)
  tibble(
    GlycanTypeSummary  = gt,
    with_qtl_in_class  = mat[1, 1],
    with_qtl_in_others = mat[1, 2],
    total_in_class     = sum(in_class),
    total_in_others    = sum(!in_class),
    qtl_rate_in_class  = mat[1, 1] / sum(in_class),
    qtl_rate_in_others = mat[1, 2] / sum(!in_class),
    odds_ratio = unname(ft$estimate),
    ci_lower   = ft$conf.int[1],
    ci_upper   = ft$conf.int[2],
    p_value    = ft$p.value
  )
}) %>%
  mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
  arrange(desc(odds_ratio))

saveRDS(main_class_enrich_gp, "data/Figure_2b.rds")
main_class_enrich_gp


## Figure 2c

In [ ]:
gp_class_long <- annotation %>%
  filter(gpID %in% used_gpid) %>%
  distinct(gpID, GlycanTypeSummary, GlycanClass) %>%
  mutate(class = str_split(GlycanClass, ";")) %>%
  unnest(class) %>%
  mutate(class = str_trim(class)) %>%
  filter(class != "") %>%
  distinct(gpID, GlycanTypeSummary, class)

### 2c: gpID-level QTL enrichment by glycan subclass (within glycosylation)

In [ ]:
gp_long_glyco <- gp_class_long %>%
  # # "Glycation" was renamed "Hexose-only" in the corrected annotation
  filter(GlycanTypeSummary == "glycosylation", !class %in% c("Glycation", "Hexose-only")) # since W;N was included with -W ignored in glycosylation

gp_set_glyco <- gp_long_glyco %>%
  group_by(gpID) %>%
  summarise(classes = list(unique(class)), .groups = "drop") %>%
  mutate(has_qtl = gpID %in% qtl_gpid_unadj) %>%
  left_join(gp_cs_count, by = "gpID") %>%
  mutate(n_cs = replace_na(n_cs, 0L))

glycan_classes <- gp_long_glyco %>% pull(class) %>% unique()

enrich_each_vs_others_gp <- map_dfr(glycan_classes, function(gc) {
  in_class <- map_lgl(gp_set_glyco$classes, ~ gc %in% .x)
  mat <- matrix(
    c(sum( in_class &  gp_set_glyco$has_qtl), sum(!in_class &  gp_set_glyco$has_qtl),
      sum( in_class & !gp_set_glyco$has_qtl), sum(!in_class & !gp_set_glyco$has_qtl)),
    nrow = 2, byrow = TRUE,
    dimnames = list(QTL = c("WithQTL","WithoutQTL"), Type = c(gc,"Others"))
  )
  ft <- fisher.test(mat)
  tibble(GlycanClass        = gc,
         with_qtl_in_class  = mat[1,1], with_qtl_in_others = mat[1,2],
         total_in_class     = mat[1,1] + mat[2,1],
         total_in_others    = mat[1,2] + mat[2,2],
         qtl_rate_in_class  = mat[1,1] / (mat[1,1] + mat[2,1]),
         qtl_rate_in_others = mat[1,2] / (mat[1,2] + mat[2,2]),
         odds_ratio = unname(ft$estimate),
         ci_lower = ft$conf.int[1], ci_upper = ft$conf.int[2], p_value = ft$p.value)
}) %>%
  mutate(p_adj = p.adjust(p_value, method = "BH")) %>%
  arrange(desc(odds_ratio))
saveRDS(enrich_each_vs_others_gp, "data/Figure_2c.rds")

enrich_each_vs_others_gp


In [ ]:
cus_colors <- setNames(
  c(brewer.pal(4, "Paired"), "grey50"),
  c("High-mannose", "Paucimannose", "Hybrid/Complex", "Glycation", "Other")
)

f2c <- ggplot(enrich_each_vs_others_gp %>% filter(!GlycanClass %in% c('Glycation', 'Hexose-only')),
              aes(x = reorder(GlycanClass, -odds_ratio), y = odds_ratio, fill = GlycanClass)) +
  geom_col(width = 0.7) +
  geom_errorbar(aes(ymin = ci_lower, ymax = ci_upper), width = 0.2, size = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed") +
  labs(x = "Glycan class", y = "With-QTL rate vs Others (odds ratio)") +
  theme_classic(base_size = 16) +
  theme(legend.position = "none", axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_manual(values = cus_colors) +
  scale_x_discrete(labels = c("Hybrid/Complex" = "Hybrid/complex"))

## Figure 2d

In [ ]:
library(tidyverse)
library(RColorBrewer)
enrichment_data <- readRDS('EOO/enrichment_figures_category/context_compare/glycation_glycosylation_eoo_enrichment.rds')
enrichment_data <- enrichment_data %>% filter(!(annotation_resource == 'AI_MPRA_starrseq' & str_detect(Annotation, 'LHX2|PU1|NeuN|Olig2')))  # remove Nott data from AI_xQTL
track_df <- fread("EOO/data/categorized_annotation_tracks.csv")

meta <- fread('EOO/data/realease_data_readme.txt')
meta <- meta %>% mutate(context_mod = basename(Path) %>% gsub('.exported.toploci.bed.gz', '',.))

# Step 1: Prepare data with initial filtering
enrichment_data_top_category <- enrichment_data %>%
  filter(
    !str_detect(Annotation, 'ENCFF'),
    str_detect(Annotation, 'CellType'),
    Priority == 'Tier 1'
  )

enrichment_data_top_category <- merge(
  enrichment_data_top_category,
  meta,
  by.x = 'study',
  by.y = 'context_mod',
  all.x = TRUE
) %>%
  mutate(
    `Data Type` = 'fine_mapping',
    `Data Type` = ifelse(str_detect(study, 'AC|PCC|DLPFC|MSBB|Knight'), 'eQTL', `Data Type`),
    `Data Type` = ifelse(str_detect(study, 'Inh|Exc|Oli|OPC|Mic|Ast|monocyte'), 'snuc_eQTL', `Data Type`),
    `Data Type` = ifelse(str_detect(study, '_pQTL'), 'pQTL', `Data Type`)
  )

plot_data_tmp <- enrichment_data_top_category %>% filter(study %in% c("glycation_unadjusted_maf05","glycosylation_unadjusted_maf05"))
plot_data_tmp$study <- factor(plot_data_tmp$study, levels = c('glycation_unadjusted_maf05','glycosylation_unadjusted_maf05'))

In [ ]:
# Step 4: Clean and reformat annotation labels
plot_data_tmp <- plot_data_tmp %>%
  mutate(
    Annotation = gsub('formatted_output_', '', Annotation),
    Annotation = gsub('CellType_', '', Annotation),
    Annotation = gsub('_lifted.bed', '', Annotation)
  ) 
plot_data_tmp$Annotation <- plot_data_tmp$Annotation %>%
  str_replace("^Astrocyte\\|LHX2_optimal_peak_IDR_ENCODE\\.ATAC$", "Astrocyte_ATAC") %>%
  str_replace("^Astrocyte\\|LHX2_optimal_peak_IDR_ENCODE\\.H3K27.*$", "Astrocyte_H3K27ac") %>%
  str_replace("^Astrocyte\\|LHX2_optimal_peak_IDR_ENCODE\\.H3K4me3$", "Astrocyte_H3K4me3") %>%
  str_replace("^Astrocyte\\|LHX2_superEnhancer_L1$", "Astrocyte_SuperEnhancer") %>%
  
  str_replace("^Neuron\\|NeuN_optimal_peak_IDR_ENCODE\\.ATAC$", "Neuron_ATAC") %>%
  str_replace("^Neuron\\|NeuN_optimal_peak_IDR_ENCODE\\.H3K27.*$", "Neuron_H3K27ac") %>%
  str_replace("^Neuron\\|NeuN_optimal_peak_IDR_ENCODE\\.H3K4me3$", "Neuron_H3K4me3") %>%
  str_replace("^Neuron\\|NEUN_superEnhancer_L1$", "Neuron_SuperEnhancer") %>%
  
  str_replace("^Oligodendrocyte\\|Olig2_optimal_peak_IDR_ENCODE\\.ATAC$", "Oligodendrocyte_ATAC") %>%
  str_replace("^Oligodendrocyte\\|Olig2_optimal_peak_IDR_ENCODE\\.H3K27.*$", "Oligodendrocyte_H3K27ac") %>%
  str_replace("^Oligodendrocyte\\|Olig2_optimal_peak_IDR_ENCODE\\.H3K4me3$", "Oligodendrocyte_H3K4me3") %>%
  str_replace("^Oligodendrocyte\\|OLIG2_superEnhancer_L1$", "Oligodendrocyte_SuperEnhancer") %>%
  
  str_replace("^Microglia_Myeloid\\|PU1_optimal_peak_IDR_ENCODE\\.ATAC$", "Microglia_ATAC") %>%
  str_replace("^Microglia_Myeloid\\|PU1_optimal_peak_IDR_ENCODE\\.H3K27.*$", "Microglia_H3K27ac") %>%
  str_replace("^Microglia_Myeloid\\|PU1_optimal_peak_IDR_ENCODE\\.H3K4me3$", "Microglia_H3K4me3") %>%
  str_replace("^Microglia_Myeloid\\|PU1_superEnhancer_L1$", "Microglia_SuperEnhancer")

plot_data_tmp <- plot_data_tmp[!str_detect(plot_data_tmp$Annotation, "PLAC|enhancers"), ]

annotation_order <- plot_data_tmp %>%
  group_by(Category, Annotation) %>%
  slice(which.max(Enrichment_log2)) %>%
  ungroup() %>%
  arrange(Category, desc(Enrichment_log2)) %>%
  pull(Annotation) %>%
  unique()

plot_data_tmp <- plot_data_tmp %>%
  mutate(Annotation = factor(Annotation, levels = annotation_order))

x_limits <- plot_data_tmp %>%
  summarise(
    min_val = min(Enrichment_log2 - Enrichment_SE_log2, na.rm = TRUE),
    max_val = max(Enrichment_log2 + Enrichment_SE_log2, na.rm = TRUE)
  )

In [ ]:
options(repr.plot.width=14, repr.plot.height=14)

plot_data_tmp <- plot_data_tmp %>%
  mutate(
    Function = case_when(
      str_detect(Annotation, "ATAC") ~ "ATAC",
      str_detect(Annotation, "H3K27ac") ~ "H3K27ac",
      str_detect(Annotation, "H3K4me3") ~ "H3K4me3",
      str_detect(Annotation, "SuperEnhancer") ~ "Super-enhancer",
      str_detect(Annotation, "Enhancer") ~ "Enhancer",
      TRUE ~ "Other"
    ),
    CellType = case_when(
      str_detect(Annotation, "^Astrocyte") ~ "Astrocyte",
      str_detect(Annotation, "^Neuron") ~ "Neuron",
      str_detect(Annotation, "^Oligodendrocyte") ~ "Oligodendrocyte",
      str_detect(Annotation, "^Microglia") ~ "Microglia",
      TRUE ~ "Other"
    )
  )

function_order <- c("ATAC", "H3K27ac", "H3K4me3", "Enhancer", "Super-enhancer", "Other")
celltype_order <- c("Astrocyte", "Neuron", "Oligodendrocyte", "Microglia", "Other")

annotation_order2 <- plot_data_tmp %>%
  distinct(Annotation, CellType, Function) %>%
  mutate(
    CellType = factor(CellType, levels = celltype_order),
    Function = factor(Function, levels = function_order)
  ) %>%
  arrange(CellType, Function, Annotation) %>%
  pull(Annotation)

plot_data_tmp <- plot_data_tmp %>%
  mutate(Annotation = factor(Annotation, levels = rev(annotation_order2)))

anno_bg <- plot_data_tmp %>%
  distinct(Annotation, CellType) %>%
  mutate(
    y_id = as.numeric(Annotation),
    ymin = y_id - 0.5,
    ymax = y_id + 0.5,
    xmin = x_limits$min_val,
    xmax = x_limits$max_val
  )

study_levels <- unique(plot_data_tmp$study)
anno_bg <- tidyr::crossing(anno_bg, study = study_levels)

function_colors <- c(
  "ATAC" = "#4E79A7",
  "H3K27ac" = "#E15759",
  "H3K4me3" = "#76B7B2",
  "Enhancer" = "#59A14F",
  "Super-enhancer" = "#B07AA1",
  "Other" = "#9C9C9C"
)

celltype_colors <- c(
  "Astrocyte" = "#FCE8E6",
  "Neuron" = "#EAF2FB",
  "Oligodendrocyte" = "#EAF6EA",
  "Microglia" = "#FFF3E0",
  "Other" = "#F3F3F3"
)

anno_bg$bg_fill <- unname(celltype_colors[anno_bg$CellType])

annotation_labels <- setNames(
  nm = annotation_order2,
  object = annotation_order2 %>%
    str_replace("^Astrocyte_", "Astrocyte: ") %>%
    str_replace("^Neuron_", "Neuron: ") %>%
    str_replace("^Oligodendrocyte_", "Oligodendrocyte: ") %>%
    str_replace("^Microglia_", "Microglia: ")
)

f2d <- ggplot(plot_data_tmp, aes(x = Enrichment_log2, y = Annotation)) +
  geom_rect(
    data = anno_bg,
    aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax),
    inherit.aes = FALSE,
    fill = anno_bg$bg_fill,
    alpha = 0.32,
    color = NA
  ) +
  geom_vline(xintercept = 0, linetype = "dashed", linewidth = 0.45, color = "grey45") +
  geom_col(
    aes(fill = Function),
    width = 0.62,
    color = "grey18",
    linewidth = 0.22
  ) +
  geom_errorbar(
    aes(
      xmin = Enrichment_log2 - Enrichment_SE_log2,
      xmax = Enrichment_log2 + Enrichment_SE_log2
    ),
    width = 0.18,
    linewidth = 0.4,
    color = "grey18"
  ) +
  facet_wrap(~study, ncol = 2) +
  coord_cartesian(xlim = c(x_limits$min_val, x_limits$max_val), clip = "off") +
  scale_fill_manual(
    values = function_colors,
    breaks = function_order,
    name = "Functional annotation"
  ) +
  scale_y_discrete(
    limits = levels(plot_data_tmp$Annotation),
    labels = annotation_labels,
    expand = expansion(mult = c(0.01, 0.01))
  ) +
  labs(
    title = "Enrichment across annotations",
    x = expression(log[2]("Enrichment score")),
    y = NULL
  ) +
  theme_classic(base_size = 14) +
  theme(
    strip.background = element_rect(fill = "grey97", color = "grey75", linewidth = 0.35),
    strip.text = element_text(face = "bold", size = 11.5, color = "black"),
    axis.text.y = element_text(size = 10.5, color = "black"),
    axis.text.x = element_text(size = 10.5, color = "black"),
    axis.title.x = element_text(size = 11.5, face = "bold"),
    plot.title = element_text(size = 14.5, face = "bold", hjust = 0.5),
    legend.position = "bottom",
    legend.title = element_text(size = 10.5, face = "bold"),
    legend.text = element_text(size = 9.8),
    legend.key.width = unit(1.2, "lines"),
    panel.grid.major.x = element_line(color = "grey90", linewidth = 0.35),
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    axis.line.y = element_blank(),
    axis.ticks.y = element_blank(),
    plot.margin = margin(8, 14, 8, 8)
  )

f2d

In [ ]:
saveRDS(plot_data_tmp, "data/Figure_2d.rds")


## Figure 2E

In [ ]:
library(tidyverse)
library(data.table)
library(UpSetR)
gp_e <- fread('data_output/colocalization/maf05/glycoQTL_maf05/glycoQTL_eQTL_summary_pv1e4.tsv')
# standard recipe: cos_npc >= 0.2, npc_outcome > 0.2, p < 1e-4
stopifnot(all(gp_e$cos_npc >= 0.2 & gp_e$min_npc_outcome > 0.2))

qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
coding_drop_gpids <- readLines('../../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt')
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids, coding_drop_gpids)
strip_excluded_gpids <- function(s) {
  parts <- trimws(strsplit(s, ";")[[1]])
  gp_in_part <- str_extract(parts, "gp_[0-9]+")
  parts[is.na(gp_in_part) | !(gp_in_part %in% excluded_gpids)]
}
suppressMessages(library(colocboost))
fix_cw <- function(cb) { cb$cos_details$cos_weights <- lapply(cb$cos_details$cos_weights, function(w) { colnames(w) <- sub("^Y", "outcome", colnames(w)); w }); cb }  # v202609 rds name cos_weights columns Y<N>; colocboost 1.0.7 expects outcome<N>
tm_rds <- 'data_output/TMEM106B_rerun_202609/glycoQTL_tmem106b/glycoQTL_unadjusted_eQTL/colocboost/eQTL_gpQTL.chr7_ENSG00000106460.cb_xqtl.rds'
tm_new <- suppressMessages(suppressWarnings(get_robust_colocalization(fix_cw(readRDS(tm_rds)[[1]]), cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 1e-4)))$cos_summary
tm_new$gene <- 'ENSG00000106460'
tm_new <- tm_new[grepl('gp_', tm_new$colocalized_outcomes), ]
cat("TMEM106B rows: old", sum(gp_e$gene == 'ENSG00000106460'), "-> new", nrow(tm_new), "\n"); print(tm_new$colocalized_outcomes)
gp_e <- rbind(gp_e[gene != 'ENSG00000106460'], tm_new[, intersect(names(gp_e), names(tm_new))], fill = TRUE)
cat("colocalized_outcomes tokens matching qc_remove_gpids:",
    sum(str_detect(gp_e$colocalized_outcomes, paste(qc_remove_gpids, collapse = "|"))), "\n")
cat("colocalized_outcomes tokens matching coding_drop_gpids:",
    sum(str_detect(gp_e$colocalized_outcomes, paste(coding_drop_gpids, collapse = "|"))), "\n")
cat("colocalized_outcomes tokens matching w_sole_exclude_gpids:",
    sum(str_detect(gp_e$colocalized_outcomes, paste(w_sole_exclude_gpids, collapse = "|"))), "\n")
trimmed_parts <- lapply(gp_e$colocalized_outcomes, strip_excluded_gpids)
keep_row <- lengths(trimmed_parts) >= 2
cat(sprintf("Rows dropped entirely (<2 members remain after stripping): %d of %d\n", sum(!keep_row), nrow(gp_e)))
gp_e <- gp_e[keep_row]
gp_e$colocalized_outcomes <- vapply(trimmed_parts[keep_row], paste, character(1), collapse = "; ")

contexts <- c("gpQTL_unadjusted", "Ast", "Mic", "Exc", "Oli", "OPC", "Inh")
coloc <- vector("list", length(contexts)) %>% setNames(contexts)
for (i in 1:nrow(gp_e)){
    traits <- gp_e$colocalized_outcomes[i]
    for (con in contexts){
        if_coloc <- grepl(con, traits)
        if (if_coloc){
            coloc[[con]] <- c(coloc[[con]], paste0("cos_", i, "_", gp_e$gene[i]))
        }
    }
}

In [ ]:
# build binary incidence matrix, rename a set, and keep only rows with >1 membership
coloc_df <- fromList(coloc) %>% 
  rename(Glyco = gpQTL_unadjusted)
coloc_df <- coloc_df[rowSums(coloc_df) > 1, , drop = FALSE] 

contexts_ordered <- names(sort(colSums(coloc_df), decreasing = TRUE))

max_size <- max(colSums(coloc_df))

saveRDS(coloc_df, 'data/Figure_2e.rds')

## Figure 2F

In [ ]:
# 2026-10: Fig 2f uses the new S-LDSC pipeline
source('../../2_main_figures/Figure2/data/LDSC_selected_traits.R')
traits_dt_glycoslation <- readRDS('SLDSC/output_v202609/new_pipeline/figures/SLDSC.seperate.plot.glycoslation.res.new_pipeline.rds') %>%
  mutate(GlycanSummary = 'Glycosylation')

traits_dt_glycoslation[, `:=`(
  ci95_lower = enrichment - 1.96 * se,
  ci95_upper = enrichment + 1.96 * se,
  ci68_lower = enrichment - 1.00 * se,
  ci68_upper = enrichment + 1.00 * se
)]

traits_dt <- traits_dt_glycoslation[enrichment > 0] %>%
  mutate(is_sig = p < 0.05) %>%
  filter(trait != 'Dementia with Lewy Bodies (Chia GCST90001390)')

traits_dt <- traits_dt %>%
  group_by(category, GlycanSummary) %>%
  mutate(trait = forcats::fct_reorder(trait, enrichment)) %>%
  ungroup()

traits_dt_hans <- traits_dt %>%
  filter(trait %in% c(Neurodegenerative_NeurologicalDisorders_Traits,
                      Psychiatric_Cognitive_Traits,
                      Brain_Imaging_Morphometry_Traits)) %>%
  mutate(category = case_when(
    trait %in% Neurodegenerative_NeurologicalDisorders_Traits ~ "Neurodegenerative / Neurological Disorders",
    trait %in% Psychiatric_Cognitive_Traits ~ "Psychiatric / Cognitive Traits",
    trait %in% Brain_Imaging_Morphometry_Traits ~ "Brain Imaging / Morphometry",
    TRUE ~ NA_character_
  )) %>%
  mutate(trait    = gsub(" \\(from Lu\\)", "", trait),
         category = as.factor(category))

traits_dt_hans$category <- factor(traits_dt_hans$category,
  levels = c("Neurodegenerative / Neurological Disorders",
             "Psychiatric / Cognitive Traits",
             "Brain Imaging / Morphometry"))

In [ ]:
# library(colorspace)    # uncomment if not already loaded

category_colors <- c(
  "Neurodegenerative / Neurological Disorders" = "#D73027",     # 
  "Brain Imaging / Morphometry" = "#4575B4",     # 
  "All" = "#1A9850",    # 
  "Psychiatric / Cognitive Traits" = "#984EA3"      #  
)


traits_dt_hans <- traits_dt_hans %>%
  group_by(category, GlycanSummary) %>%
  mutate(
    trait_order = as.integer(factor(trait, levels = unique(trait[order(enrichment)]))),
    n_traits = n(),
    color = colorspace::lighten(category_colors[category],
                                amount = scales::rescale(trait_order, to = c(0.6, 0.1)))
  ) %>%
  ungroup()

traits_dt_hans <- traits_dt_hans %>%
  mutate(
    GlycanSummary = factor(GlycanSummary, levels = c("Glycation", "Glycosylation")),
    trait_in_panel = reorder_within(trait, enrichment, interaction(GlycanSummary, category))
  )

traits_dt_hans <- traits_dt_hans %>%
  mutate(
    trait_in_panel = fct_reorder(trait_in_panel, enrichment, .fun = identity),
    trait_in_panel = fct_reorder(trait_in_panel, -as.numeric(factor(category))),
      category = gsub('/', '/\n', category), category = factor(category, levels=   c("Neurodegenerative /\n Neurological Disorders",  "Psychiatric /\n Cognitive Traits", "Brain Imaging /\n Morphometry"))
  )

saveRDS(traits_dt_hans, 'data/Figure_2f.rds')

In [ ]:
# Panel F (main figure): standardized S-LDSC coefficient tau* (replaces enrichment)
library(data.table); library(tidytext)
M <- readRDS("SLDSC/output_v202609/new_pipeline/step2/output/ROSMAP_gpQTL_glycosylation_unadjusted_maf05/all_m/postprocess/ROSMAP_gpQTL_glycosylation_unadjusted_maf05.sldsc_postprocess.rds")$params$M_ref
cat(sprintf("M (new pipeline M_ref): %d\n", M))
stopifnot(M > 8e6, M < 8.3e6)

f2f_dt <- as.data.table(traits_dt_hans)
f2f_dt <- f2f_dt[h2g > 0]                       # tau* undefined for non-positive h2g
f2f_dt[, `:=`(tau_star = coef * sd_annot * M / h2g, taus_se = coef_se * sd_annot * M / h2g,
              coef_p2 = 2 * pnorm(-abs(coef_z)))]
f2f_dt[, `:=`(taus_lo = tau_star - 1.96 * taus_se, taus_hi = tau_star + 1.96 * taus_se, sig = coef_p2 < 0.05)]
f2f_dt[, trait_in_panel := droplevels(trait_in_panel)]
saveRDS(list(data = f2f_dt, excluded = setdiff(as.character(traits_dt_hans$trait), as.character(f2f_dt$trait))),
        "data/Figure_2f_taustar.rds")

f2f_plot <- f2f_dt[trait != "Alzheimer's Disease (Jansen 2019)"]
f2f_plot[, trait_in_panel := droplevels(trait_in_panel)]

f2f <- ggplot(f2f_plot, aes(x = tau_star, y = trait_in_panel, color = color)) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey50") +
  geom_point(size = 3) +
  geom_errorbarh(aes(xmin = taus_lo, xmax = taus_hi), height = 0.3) +
  geom_text(data = f2f_plot[sig == TRUE],
            aes(x = taus_hi, label = ifelse(coef_p2 < 0.001, "***", ifelse(coef_p2 < 0.01, "**", "*"))),
            size = 7, hjust = -0.2, vjust = 0.75, show.legend = FALSE) +
  scale_y_reordered() + scale_color_identity() +
  facet_grid(category ~ ., scales = "free_y", space = "free_y") +
  labs(x = expression("Standardized effect size "*tau*"* (95% CI)"), y = "Trait") +
  theme_minimal(base_size = 16) +
  theme(strip.text = element_text(face = "bold", size = 14),
        axis.text = element_text(size = 14), axis.title = element_text(size = 14),
        panel.grid.minor = element_blank(), panel.grid.major.y = element_blank(),
        strip.placement = "outside")


## Fig 2f: S-LDSC tau* and coefficient p-values for highlighted traits

In [ ]:
library(data.table)
library(tidyverse)

f2f_dt <- as.data.table(readRDS("data/Figure_2f_taustar.rds"))
highlight <- c("Parkinson's Disease (Nalls 2019 excl. 23andMe)",
               "Alzheimer's Disease (Kunkle Stage 1)",
               "Alzheimer's Disease (Bellenguez)",
               "Language", "Memory")
print(f2f_dt[data.trait %in% highlight,
             .(data.trait, enrichment = round(data.enrichment, 1), tau_star = round(data.tau_star, 2), coef_p = signif(data.coef_p2, 3))][order(-enrichment)])